In [1]:
# ============================
# SU(3) Wilson plaquette: "erosion constant" from the *correct* r^2 mechanism
# (kernel/range split + Schur complement), in the exponential-coordinate chart.
# ============================

import numpy as np
import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)

# ----------------------------
# Tunables
# ----------------------------
NORMALIZE_TRACE = True     # True matches 1 - (1/3) Re Tr(U_p)
EXP_METHOD = "pade22"      # "pade22" matches many fast scan codes; OK for derivatives up to 4th at 0
KERNEL_EPS = 1e-10
N_PROBES = 128             # random directions to probe for sup estimates (increase for tighter)
TEST_RS = [1e-3, 3e-3, 1e-2]  # small radii for direct Hessian sanity checks

# Choose the amplitude statistic r. This should match what you WANT in the theorem.
# Here: r = max over the 4 links of the 8-vector Euclidean norm (linkwise L2, then L∞ over links).
def r_linkL2_inf(x32):
    x = x32.reshape((4,8))
    return jnp.max(jnp.linalg.norm(x, axis=1))

def normalize_direction(w):
    return w / r_linkL2_inf(w)

# ----------------------------
# SU(3) basis
# ----------------------------
def gell_mann():
    lm = []
    lm.append(jnp.array([[0,1,0],[1,0,0],[0,0,0]], dtype=jnp.complex128))
    lm.append(jnp.array([[0,-1j,0],[1j,0,0],[0,0,0]], dtype=jnp.complex128))
    lm.append(jnp.array([[1,0,0],[0,-1,0],[0,0,0]], dtype=jnp.complex128))
    lm.append(jnp.array([[0,0,1],[0,0,0],[1,0,0]], dtype=jnp.complex128))
    lm.append(jnp.array([[0,0,-1j],[0,0,0],[1j,0,0]], dtype=jnp.complex128))
    lm.append(jnp.array([[0,0,0],[0,0,1],[0,1,0]], dtype=jnp.complex128))
    lm.append(jnp.array([[0,0,0],[0,0,-1j],[0,1j,0]], dtype=jnp.complex128))
    lm.append((1/jnp.sqrt(3))*jnp.array([[1,0,0],[0,1,0],[0,0,-2]], dtype=jnp.complex128))
    return jnp.stack(lm, axis=0)

LAM = gell_mann()
T = 0.5j * LAM   # anti-Hermitian basis; -Tr(T_a T_b) = δ_ab/2

def vec_to_alg(v8):
    # v8 real (8,)
    return jnp.tensordot(v8, T, axes=1)  # (3,3) complex anti-Hermitian

# ----------------------------
# exp map (fast + differentiable)
# ----------------------------
def exp_pade22(A):
    I = jnp.eye(3, dtype=jnp.complex128)
    A2 = A @ A
    Num = I + 0.5*A + (1.0/12.0)*A2
    Den = I - 0.5*A + (1.0/12.0)*A2
    # Den^{-1} Num
    return jnp.linalg.solve(Den, Num)

def exp_series(A, order=12):
    I = jnp.eye(3, dtype=jnp.complex128)
    out = I
    term = I
    for k in range(1, order+1):
        term = term @ (A / k)
        out = out + term
    return out

def su3_exp(A):
    if EXP_METHOD == "pade22":
        return exp_pade22(A)
    elif EXP_METHOD == "series":
        return exp_series(A, order=12)
    else:
        raise ValueError("EXP_METHOD must be 'pade22' or 'series'")

# ----------------------------
# One-plaquette Wilson action in the exponential chart
# U_p = exp(A1) exp(A2) exp(-A3) exp(-A4)
# S_p = 1 - (1/3) Re Tr(U_p)
# ----------------------------
def plaquette_action(x32):
    x = x32.reshape((4,8))
    A1, A2, A3, A4 = [vec_to_alg(x[i]) for i in range(4)]
    U1 = su3_exp(A1)
    U2 = su3_exp(A2)
    U3 = su3_exp(-A3)
    U4 = su3_exp(-A4)
    Up = U1 @ U2 @ U3 @ U4
    tr = jnp.trace(Up)
    if NORMALIZE_TRACE:
        tr = tr / 3.0
    return 1.0 - jnp.real(tr)  # real scalar

Hess = jax.hessian(plaquette_action)

x0 = jnp.zeros((32,), dtype=jnp.float64)
H0 = Hess(x0)
H0 = 0.5*(H0 + H0.T)  # symmetrize

evals, evecs = jnp.linalg.eigh(H0)
ker_mask = evals < KERNEL_EPS
K = evecs[:, ker_mask]      # (32,k)
R = evecs[:, ~ker_mask]     # (32,8) for SU(3) plaquette in this chart
lam_pos = evals[~ker_mask]
Pinv = jnp.diag(1.0/lam_pos)

print("=== Wilson Hessian at 0 (one plaquette) ===")
print("min eig(H0) =", float(evals[0]), "| max eig(H0) =", float(evals[-1]))
print("kernel dim k =", int(K.shape[1]), " | range dim =", int(R.shape[1]))
print("positive eigs (should be 8 of them):", np.array(lam_pos))

# ----------------------------
# Directional derivatives of the Hessian at 0 WITHOUT building full D3/D4.
# dH(w)  = d/dt Hess(x0 + t w)|0  (D3 contracted with w)
# ddH(w) = d^2/dt^2 Hess(x0 + t w)|0 (D4 contracted with w,w)
# ----------------------------
def dH_at0(w):
    _, dH = jax.jvp(Hess, (x0,), (w,))
    dH = 0.5*(dH + dH.T)
    return dH

def ddH_at0(w):
    def H1(x):
        _, dH = jax.jvp(Hess, (x,), (w,))
        return dH
    _, ddH = jax.jvp(H1, (x0,), (w,))
    ddH = 0.5*(ddH + ddH.T)
    return ddH

# Linear-in-r kernel block: A(w) = (dH)_KK
def kernel_linear_opnorm(w):
    w = normalize_direction(w)
    dH = dH_at0(w)
    A = K.T @ dH @ K
    A = 0.5*(A + A.T)
    eigsA = jnp.linalg.eigvalsh(A)
    return jnp.max(jnp.abs(eigsA))

# The "correct" r^2 kernel effective matrix:
# Heff2(w) = 0.5*(ddH)_KK  - (dH)_KR Pinv (dH)_RK
def Heff2_on_kernel(w):
    w = normalize_direction(w)
    dH  = dH_at0(w)
    ddH = ddH_at0(w)
    dH_KR  = K.T @ dH  @ R           # (k,8)
    ddH_KK = K.T @ ddH @ K           # (k,k)
    schur = dH_KR @ Pinv @ dH_KR.T   # (k,k)
    Heff2 = 0.5*ddH_KK - schur
    Heff2 = 0.5*(Heff2 + Heff2.T)
    return Heff2

def C_dir(w):
    # Directional erosion constant for one plaquette, in the chosen r-norm:
    # Hess(r w) has a most-negative mode ~ - C_dir * r^2 (for small r).
    Heff2 = Heff2_on_kernel(w)
    eigs = jnp.linalg.eigvalsh(Heff2)
    return jnp.maximum(0.0, -eigs[0])

# ----------------------------
# Probe random directions
# ----------------------------
key = jax.random.PRNGKey(0)
W = jax.random.normal(key, shape=(N_PROBES, 32), dtype=jnp.float64)

C_vals = jax.vmap(C_dir)(W)
A_lin_vals = jax.vmap(kernel_linear_opnorm)(W)

print("\n=== Probed directions (normalized by r_linkL2_inf=1) ===")
print("median ||(dH)_KK||_op  =", float(jnp.median(A_lin_vals)))
print("max    ||(dH)_KK||_op  =", float(jnp.max(A_lin_vals)))
print("median C_dir (plaquette) =", float(jnp.median(C_vals)))
print("max    C_dir (plaquette) =", float(jnp.max(C_vals)))

C_plaq_est = float(jnp.max(C_vals))
C_link_est = 6.0 * C_plaq_est   # 4D: each link participates in 6 plaquettes
print("\n=== Estimated constants ===")
print("C_plaq_est  (1 plaquette) =", C_plaq_est)
print("C_link_est  (×6 plaquettes/link, 4D) =", C_link_est)

# ----------------------------
# Sanity check vs direct Hessian at small r along a few directions
# ----------------------------
def lambda_min_W(r, w):
    x = r * normalize_direction(w)
    H = Hess(x)
    H = 0.5*(H + H.T)
    return jnp.linalg.eigvalsh(H)[0]

print("\n=== Small-r sanity check (Wilson-only) ===")
for i in range(3):
    w = W[i]
    Cpred = float(C_dir(w))
    print(f"\nDirection {i}: C_pred_dir≈{Cpred:.6g}")
    for r in TEST_RS:
        lam = float(lambda_min_W(r, w))
        Cemp = -lam/(r*r)
        print(f"  r={r: .1e}  lambda_min≈{lam:+.3e}  (-lambda_min/r^2)≈{Cemp:.6g}")

print("\nDone.")


/usr/local/lib/python3.12/dist-packages/jax/_src/lax/lax.py:5473: ComplexWarning: Casting complex values to real discards the imaginary part
  x_bar = _convert_element_type(x_bar, x.aval.dtype, x.aval.weak_type)


=== Wilson Hessian at 0 (one plaquette) ===
min eig(H0) = -8.865583466863452e-16 | max eig(H0) = 0.666666666666667
kernel dim k = 24  | range dim = 8
positive eigs (should be 8 of them): [0.66666667 0.66666667 0.66666667 0.66666667 0.66666667 0.66666667
 0.66666667 0.66666667]

=== Probed directions (normalized by r_linkL2_inf=1) ===
median ||(dH)_KK||_op  = 0.13152320298521908
max    ||(dH)_KK||_op  = 0.22220393250457482
median C_dir (plaquette) = 0.04846074383900344
max    C_dir (plaquette) = 0.10869967929862502

=== Estimated constants ===
C_plaq_est  (1 plaquette) = 0.10869967929862502
C_link_est  (×6 plaquettes/link, 4D) = 0.6521980757917502

=== Small-r sanity check (Wilson-only) ===

Direction 0: C_pred_dir≈0.0345614
  r= 1.0e-03  lambda_min≈-1.116e-04  (-lambda_min/r^2)≈111.604
  r= 3.0e-03  lambda_min≈-3.350e-04  (-lambda_min/r^2)≈37.219
  r= 1.0e-02  lambda_min≈-1.118e-03  (-lambda_min/r^2)≈11.1842

Direction 1: C_pred_dir≈0.0509715
  r= 1.0e-03  lambda_min≈-1.371e-04  (-lamb